# Weapon Detection — YOLOv11s Training

**Primary training notebook** for the AI-Powered Weapon Detection project.

- Model: YOLOv11s (9.4M params)
- Classes: `handgun`, `long_gun`, `knife`, `explosive`
- Runtime: Google Colab with T4 GPU

## Prerequisites
1. Run `prepare_dataset.py` locally to create the merged dataset
2. Zip the `weapon_detection_v2/` directory
3. Upload to Google Drive or directly to Colab

In [ ]:
# Cell 1: Setup — install dependencies and mount Drive
!pip install -q ultralytics>=8.3.0

# Mount Google Drive (if this fails, restart runtime and try again)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Upload and extract dataset
#
# Option A: From Google Drive (recommended for large datasets)
# Upload weapon_detection_v2.zip to your Drive first
!cp /content/drive/MyDrive/weapon_detection_v2.zip /content/
!unzip -q /content/weapon_detection_v2.zip -d /content/datasets/

# Option B: Direct upload (uncomment below, comment out Option A)
# from google.colab import files
# uploaded = files.upload()  # upload weapon_detection_v2.zip
# !unzip -q weapon_detection_v2.zip -d /content/datasets/

# Verify dataset structure
!echo "=== Dataset structure ==="
!find /content/datasets/weapon_detection_v2 -type d
!echo "\n=== File counts ==="
!echo "Train images:" && ls /content/datasets/weapon_detection_v2/images/train | wc -l
!echo "Val images:" && ls /content/datasets/weapon_detection_v2/images/val | wc -l
!echo "Test images:" && ls /content/datasets/weapon_detection_v2/images/test | wc -l

In [ ]:
# Cell 3: Create dataset YAML and train
# Training saves to Google Drive so weights survive runtime disconnects

dataset_yaml = """
path: /content/datasets/weapon_detection_v2
train: images/train
val: images/val
test: images/test

nc: 4
names: ['handgun', 'long_gun', 'knife', 'explosive']
"""

with open('/content/dataset.yaml', 'w') as f:
    f.write(dataset_yaml)

from ultralytics import YOLO
from pathlib import Path

# Save directly to Google Drive to survive disconnects
save_dir = '/content/drive/MyDrive/weapon_detection_training'

model = YOLO('yolo11s.pt')

results = model.train(
    data='/content/dataset.yaml',
    epochs=120,
    patience=20,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    warmup_epochs=5,
    amp=True,
    workers=2,
    # Augmentation
    mosaic=1.0,
    scale=0.5,
    degrees=10.0,
    mixup=0.1,
    fliplr=0.5,
    flipud=0.0,
    # Saving — saves to Drive so disconnects won't lose weights
    save=True,
    save_period=10,
    project=save_dir,
    name='weapon_detector_v2',
)

print(f"\nTraining complete! Weights saved to: {save_dir}/weapon_detector_v2/")

In [ ]:
# Cell 4: Evaluate model
from ultralytics import YOLO
from pathlib import Path

# Load best weights from Drive
best_path = Path('/content/drive/MyDrive/weapon_detection_training/weapon_detector_v2/weights/best.pt')
if not best_path.exists():
    # Fallback to local path
    best_path = Path(results.save_dir) / 'weights' / 'best.pt'
model = YOLO(str(best_path))
print(f"Loaded model from: {best_path}")

# Validate on test set
test_results = model.val(data='/content/dataset.yaml', split='test')

classes = ['handgun', 'long_gun', 'knife', 'explosive']
print('\n' + '=' * 60)
print('TEST SET RESULTS')
print('=' * 60)
print(f'  mAP@50:    {test_results.box.map50:.4f}')
print(f'  mAP@50-95: {test_results.box.map:.4f}')
print(f'\n{"Class":>12s} {"Precision":>10s} {"Recall":>10s} {"mAP@50":>10s}')
print('-' * 45)
for i, cls in enumerate(classes):
    if i < len(test_results.box.ap50):
        p = test_results.box.p[i] if i < len(test_results.box.p) else 0
        r = test_results.box.r[i] if i < len(test_results.box.r) else 0
        ap50 = test_results.box.ap50[i]
        print(f'  {cls:>10s} {p:>10.4f} {r:>10.4f} {ap50:>10.4f}')

# Display training curves
from IPython.display import Image, display
results_dir = best_path.parent.parent
for plot in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    plot_path = results_dir / plot
    if plot_path.exists():
        print(f'\n{plot}:')
        display(Image(filename=str(plot_path), width=800))

In [ ]:
# Cell 5: Download best.pt
from pathlib import Path
import shutil

best_path = Path(results.save_dir) / 'weights' / 'best.pt'

# Copy to Google Drive for persistence
drive_dest = Path('/content/drive/MyDrive/weapon_detection_models')
drive_dest.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_path, drive_dest / 'best.pt')
print(f'Model saved to Google Drive: {drive_dest / "best.pt"}')

# Download to local machine
from google.colab import files
files.download(str(best_path))

In [ ]:
# Cell 6 (Optional): Export to OpenVINO for faster Intel Arc inference
#
# If you want to run inference locally on your Intel Arc 140V GPU,
# exporting to OpenVINO format gives significant speedups.

from ultralytics import YOLO
from pathlib import Path

best_path = Path(results.save_dir) / 'weights' / 'best.pt'
model = YOLO(str(best_path))

# Export to OpenVINO
model.export(format='openvino', imgsz=640, half=True)
print('OpenVINO export complete!')
print('Copy the exported _openvino_model/ directory to your local machine.')
print('Load with: model = YOLO("best_openvino_model/")')